# 1. Imports

In [2]:
# install required libraries
!pip install xlrd scikit-learn tensorflow -q

In [3]:
import os #if you are running locally or VScode, run this

FILEPATH = 'Scats Data October 2006.xls'
print('Current directory:', os.getcwd())
print('File found:', os.path.exists(FILEPATH))

Current directory: /content
File found: False


In [4]:
from google.colab import files #if you are running on colab, run this
uploaded = files.upload()
FILEPATH = list(uploaded.keys())[0]
print('Using:', FILEPATH) #upload Scats Data October 2006.xls

Saving Scats Data October 2006.xls to Scats Data October 2006.xls
Using: Scats Data October 2006.xls


In [16]:
import xlrd # reads old .xls Excel files
import numpy as np
import pandas as pd
import os
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Conv1D, MaxPooling1D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import pickle, datetime

LOOKBACK = 12
HORIZON = 1
TEST_DAYS = 6
EPOCHS = 50
BATCH = 64

print('TF version:', tf.__version__)

TF version: 2.19.0


In [6]:
def load_raw(filepath):
    # opens the XLS file and reads every row from the Data sheet
    # each row = one detector at one intersection on one day
    # with 96 traffic readings (V00-V95), one per 15-min interval
    wb = xlrd.open_workbook(filepath)
    sh = wb.sheet_by_name('Data')
    rows = []
    for r in range(2, sh.nrows):
        row = sh.row_values(r)
        dt = xlrd.xldate_as_datetime(row[9], wb.datemode).date()
        rows.append({
            'scats_id': str(row[0]).strip(),
            'location': str(row[1]).strip(),
            'lat': float(row[3]),
            'lon': float(row[4]),
            'date': dt,
            'readings': [float(v) for v in row[10:106]]
        })
    return rows

In [7]:
# load all raw rows from the XLS file
rows = load_raw(FILEPATH)

# basic dataset overview
print(f'Total detector-day records: {len(rows)}')
print(f'\nFirst record:')
print(f'SCATS ID: {rows[0]["scats_id"]}')
print(f'Location: {rows[0]["location"]}')
print(f'Lat/Lon: {rows[0]["lat"]}, {rows[0]["lon"]}')
print(f'Date: {rows[0]["date"]}')
print(f'First 6 readings (vehicles/15min): {rows[0]["readings"][:6]}')

# summary stats across the whole dataset
print(f'\nUnique sites: {len(set(r["scats_id"] for r in rows))}')
print(f'Date range: {min(r["date"] for r in rows)} to {max(r["date"] for r in rows)}')
print(f'Max flow in dataset: {max(v for r in rows for v in r["readings"])} vehicles/15min')
print(f'Avg flow: {sum(v for r in rows for v in r["readings"]) / sum(len(r["readings"]) for r in rows):.1f} vehicles/15min')

Total detector-day records: 4192

First record:
SCATS ID: 0970
Location: WARRIGAL_RD N of HIGH STREET_RD
Lat/Lon: -37.86703, 145.09159
Date: 2006-10-01
First 6 readings (vehicles/15min): [86.0, 83.0, 52.0, 58.0, 59.0, 44.0]

Unique sites: 40
Date range: 2006-10-01 to 2006-10-31
Max flow in dataset: 695.0 vehicles/15min
Avg flow: 104.0 vehicles/15min


In [12]:
def build_aggregated_df(rows):
    # each intersection has up to 4 detectors (one per road direction: N, S, E, W)
    # expand the 96 readings per row into individual timestep records
    # then sum all detector directions to get total flow per intersection per timestep
    records = []
    for row in rows:
        for i, val in enumerate(row['readings']):
            records.append({
                'scats_id': row['scats_id'],
                'lat': row['lat'],
                'lon': row['lon'],
                'date': row['date'],
                'interval': i,
                'flow': val
            })
    df = pd.DataFrame(records)
    # sum flow across all directions, average the lat/lon per site
    flow_agg = df.groupby(['scats_id', 'date', 'interval'])['flow'].sum().reset_index()
    coord_agg = df.groupby('scats_id')[['lat', 'lon']].mean().reset_index()
    agg = flow_agg.merge(coord_agg, on='scats_id')
    agg = agg.sort_values(['scats_id', 'date', 'interval']).reset_index(drop=True)
    return agg

In [13]:
# expand rows into timesteps and sum all detector directions per intersection
agg = build_aggregated_df(rows)

# peek at the aggregated data
print(f'Shape: {agg.shape}')
print(f'\nFirst 10 rows:')
print(agg.head(10).to_string())
print(f'\nFlow stats after aggregating all directions:')
print(agg['flow'].describe().round(2))

Shape: (116160, 6)

First 10 rows:
  scats_id        date  interval   flow        lat         lon
0     0970  2006-10-01         0  262.0 -37.867303  145.091511
1     0970  2006-10-01         1  248.0 -37.867303  145.091511
2     0970  2006-10-01         2  199.0 -37.867303  145.091511
3     0970  2006-10-01         3  176.0 -37.867303  145.091511
4     0970  2006-10-01         4  177.0 -37.867303  145.091511
5     0970  2006-10-01         5  142.0 -37.867303  145.091511
6     0970  2006-10-01         6  119.0 -37.867303  145.091511
7     0970  2006-10-01         7  101.0 -37.867303  145.091511
8     0970  2006-10-01         8   85.0 -37.867303  145.091511
9     0970  2006-10-01         9   78.0 -37.867303  145.091511

Flow stats after aggregating all directions:
count    116160.00
mean        360.24
std         277.16
min           0.00
25%         105.00
50%         325.00
75%         573.00
max        1409.00
Name: flow, dtype: float64


In [18]:
def make_sequences(values, dates, lookback, horizon):
    # sliding window: use `lookback` steps to predict next `horizon` steps
    # skip any window that spans a gap in dates to avoid corrupted sequences
    X, y = [], []
    for i in range(len(values) - lookback - horizon + 1):
        window_dates = dates[i:i + lookback + horizon]
        has_gap = False
        for j in range(1, len(window_dates)):
            if (window_dates[j] - window_dates[j-1]).days > 1:
                has_gap = True
                break
        if not has_gap:
            X.append(values[i:i + lookback])
            y.append(values[i + lookback:i + lookback + horizon])
    return np.array(X), np.array(y)

# peek at sequences for one site
site_group = agg[agg['scats_id'] == '0970']
sample_values = site_group['flow'].values.astype(np.float32)
sample_dates = [datetime.date.fromisoformat(str(d)) for d in site_group['date'].values]
date_per_step = []
for d in site_group['date'].unique():
    date_per_step.extend([datetime.date.fromisoformat(str(d))] * 96)

X_sample, y_sample = make_sequences(sample_values, date_per_step, LOOKBACK, HORIZON)
print(f'Site 0970 sequences: X={X_sample.shape}, y={y_sample.shape}')
print(f'\nFirst sequence input (12 steps = 3 hours):')
print(X_sample[0])
print(f'\nFirst sequence target (next 15 min):')
print(y_sample[0])

Site 0970 sequences: X=(2964, 12), y=(2964, 1)

First sequence input (12 steps = 3 hours):
[262. 248. 199. 176. 177. 142. 119. 101.  85.  78.  75.  74.]

First sequence target (next 15 min):
[74.]


In [19]:
def build_dataset(filepath, lookback=12, horizon=1, test_days=6):
    print('Loading raw data...')
    rows = load_raw(filepath)
    print(f'Loaded {len(rows)} detector-day records')

    agg = build_aggregated_df(rows)

    # time-based split: last 6 days = test, everything before = train
    # we do this BEFORE making sequences to prevent future data leaking into training
    all_dates = sorted(agg['date'].unique())
    cutoff = all_dates[-test_days]
    print(f'Train: {all_dates[0]} to {all_dates[-test_days-1]}, Test: {cutoff} to {all_dates[-1]}')

    train_X_all, train_y_all, test_X_all, test_y_all = [], [], [], []
    scalers, coords = {}, {}

    for site_id, group in agg.groupby('scats_id'):
        # store lat/lon for each site, used later to calculate road distances
        coords[site_id] = (group['lat'].iloc[0], group['lon'].iloc[0])

        values = group['flow'].values.reshape(-1, 1).astype(np.float32)

        # assign each reading its actual date (96 readings per day)
        date_per_step = []
        for d in group['date'].unique():
            date_per_step.extend([datetime.date.fromisoformat(str(d))] * 96)
        date_per_step = date_per_step[:len(values)]

        # normalise each site independently to 0-1
        scaler = MinMaxScaler(feature_range=(0, 1))
        scaled = scaler.fit_transform(values).flatten()
        scalers[site_id] = scaler

        # split by cutoff date
        train_mask = [d < cutoff for d in date_per_step]
        test_mask = [d >= cutoff for d in date_per_step]
        train_vals = scaled[train_mask]
        train_dates = [d for d, m in zip(date_per_step, train_mask) if m]
        test_vals = scaled[test_mask]
        test_dates = [d for d, m in zip(date_per_step, test_mask) if m]

        Xt, yt = make_sequences(train_vals, train_dates, lookback, horizon)
        Xv, yv = make_sequences(test_vals, test_dates, lookback, horizon)

        if len(Xt) > 0:
            train_X_all.append(Xt)
            train_y_all.append(yt)
        if len(Xv) > 0:
            test_X_all.append(Xv)
            test_y_all.append(yv)

    # combine all sites and reshape to (samples, timesteps, features) for LSTM/GRU
    X_train = np.concatenate(train_X_all).reshape(-1, lookback, 1)
    y_train = np.concatenate(train_y_all)
    X_test = np.concatenate(test_X_all).reshape(-1, lookback, 1)
    y_test = np.concatenate(test_y_all)

    print(f'X_train: {X_train.shape}, X_test: {X_test.shape}')
    print(f'Sites: {len(scalers)}, zero values in raw data: {(agg["flow"]==0).sum()}')
    return X_train, X_test, y_train, y_test, scalers, coords, agg

# run the full preprocessing pipeline
X_train, X_test, y_train, y_test, scalers, coords, agg = build_dataset(FILEPATH, LOOKBACK, HORIZON, TEST_DAYS)

Loading raw data...
Loaded 4192 detector-day records
Train: 2006-10-01 to 2006-10-25, Test: 2006-10-26 to 2006-10-31
X_train: (94056, 12, 1), X_test: (21120, 12, 1)
Sites: 40, zero values in raw data: 94
